In [1]:
from pathlib import Path
import pandas as pd
from pymatgen.io.vasp.outputs import Vasprun

def parse_vasp(path: Path | str):
    path = Path(path)
    vrun = Vasprun(filename=path)
    return vrun.final_structure, vrun.final_energy

In [2]:
records = {}
for folder in Path("mp_20_finished").iterdir():
    if folder.is_dir():
        structure, energy = parse_vasp(folder / "vasprun.xml")
        records[folder.name] = {
            "energy": energy,
            "structure": structure,
        }
data = pd.DataFrame.from_dict(records, orient="index")

In [3]:
import json
data['structure_json'] = data.structure.apply(lambda x: json.dumps(x.as_dict()))

In [5]:
data.loc[:, ['energy', 'structure_json']].to_csv("mp_20_test_pilot.csv.gz", index_label="material_id")

In [15]:
from pymatgen.io.cif import CifWriter

In [17]:
str(CifWriter(data.structure.iloc[0]))

"# generated using pymatgen\ndata_EuCuO3\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.85718541\n_cell_length_b   3.85718541\n_cell_length_c   3.85718541\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   EuCuO3\n_chemical_formula_sum   'Eu1 Cu1 O3'\n_cell_volume   57.38673892\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Eu  Eu0  1  0.50000000  0.50000000  0.50000000  1\n  Cu  Cu1  1  0.00000000  0.00000000  0.00000000  1\n  O  O2  1  0.50000000  -0.00000000  -0.00000000  1\n  O  O3  1  -0.00000000  0.50000000  -0.00000000  1\n  O  O4  1  0.00000000  -0.00000000  0.50000000  1\n"

In [ ]:
from pymatgen.core.structure import Structure
from functools import partial
data_pd = pd.read_csv("mp_20_test_pilot.csv.gz", index_col="material_id")
from_json_str = partial(Structure.from_str, fmt="json")
data_pd['structure'] = data_pd['structure_json'].apply(from_json_str)

In [14]:
data_pd

,energy,structure_json,structure
material_id,,,
mp-1075902,-36.906464,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[1.92859271 1.92859271 1.92859271] Eu, [0. 0...."
mp-12052,-74.033880,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[-0.54143974 0.93780114 3.56135458] Er, [1...."
mp-1207207,-30.201796,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[0. 0. 0.] Rb, [ 1.26070058 1.36155737 -4.54..."
mp-1216638,-33.034245,"{""@module"": ""pymatgen.core.structure"", ""@class...",[[5.14660080e-08 2.90992151e+00 0.00000000e+00...
mp-1217529,-21.691357,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[3.31349992 3.8320473 3.05919074] Tb, [2.751..."
mp-1224010,-20.936572,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[4.48186867 5.18938346 1.67596908] Hg, [2.492..."
mp-1660,-6.323225,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[0. 0. 0.] Er, [1.7707945 1.7707945 1.7707945..."
mp-567981,-26.951303,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[0. 0. 0.] Cr, [2.33653895 1.65218253 4.04700..."
mp-755570,-24.104977,"{""@module"": ""pymatgen.core.structure"", ""@class...","[[0. 0. 0.] Li, [6.61492185e-07 2.13709915e+00..."


In [19]:
from mp_api.client import MPRester
from dotenv import dotenv_values
config = dotenv_values()

In [52]:
reference_structures = []
reference_energies = []
with MPRester(config["MP_API_KEY"]) as mpr:
    for mp_id, record in records.items():
        reference_structure = mpr.get_structure_by_material_id(mp_id)
        reference_records = mpr.get_entry_by_material_id(mp_id)
        for record in reference_records:
            if record.data['run_type'] == "GGA":
                reference_energies.append(record.energy)
                break
        else:
            reference_energies.append(None)
        reference_structures.append(reference_structure)
data["reference_energy"] = reference_energies
data["reference_structure"] = reference_structures

Retrieving ThermoDoc documents: 100%|██████████| 2/2 [00:00<00:00, 41943.04it/s]


In [54]:
mp_20 = pd.read_csv("/home/kna/WyckoffTransformer/cdvae/data/mp_20/test.csv", index_col="material_id")

In [ ]:
data

ValueError: Format not specified and could not infer from filename='cif'

In [60]:
mp_20

,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number
material_id,,,,,,,,
mp-10009,6000,-0.575092,0.8980,GaTe,0.000000,"['Ga', 'Te']",# generated using pymatgen\ndata_GaTe\n_symmet...,194
mp-1218989,37702,-0.942488,0.0000,SmThCN,0.044109,"['C', 'N', 'Sm', 'Th']",# generated using pymatgen\ndata_SmThCN\n_symm...,160
mp-1225695,42245,0.064863,0.0000,CuNi,0.064863,"['Cu', 'Ni']",# generated using pymatgen\ndata_CuNi\n_symmet...,65
mp-1220884,780,-1.456116,0.0000,NaTiVS4,0.000000,"['Na', 'S', 'Ti', 'V']",# generated using pymatgen\ndata_NaTiVS4\n_sym...,8
mp-1224266,35749,0.024139,0.0000,Ho3TmMn8,0.036496,"['Ho', 'Mn', 'Tm']",# generated using pymatgen\ndata_Ho3TmMn8\n_sy...,8
...,...,...,...,...,...,...,...,...
mp-21084,7763,-1.777752,1.0134,In6Ga2PtO8,0.000000,"['Ga', 'In', 'O', 'Pt']",# generated using pymatgen\ndata_In6Ga2PtO8\n_...,225
mp-571486,15377,-0.359373,0.0000,CuSe,0.000000,"['Cu', 'Se']",# generated using pymatgen\ndata_CuSe\n_symmet...,63
mp-14410,17730,-1.205080,0.4594,Tl6TeO12,0.000000,"['O', 'Te', 'Tl']",# generated using pymatgen\ndata_Tl6TeO12\n_sy...,148


In [53]:
data

,energy,structure,reference_structure,reference_energy
mp-1075902,-36.906464,"[[1.92859271 1.92859271 1.92859271] Eu, [0. 0....","[[1.9129675 1.9129675 1.9129675] Eu, [0. 0. 0....",-39.119562
mp-12052,-74.033880,"[[-0.54143974 0.93780114 3.56135458] Er, [1....","[[0.53638263 0.929028 0.37922233] Er, [4.487...",-75.661432
mp-1207207,-30.201796,"[[0. 0. 0.] Rb, [ 1.26070058 1.36155737 -4.54...","[[0. 0. 0.] Rb, [ 4.37930250e-17 -4.37930250e-...",-30.247908
mp-1216638,-33.034245,[[5.14660080e-08 2.90992151e+00 0.00000000e+00...,"[[2.511039 1.4497519 0. ] Tm, [ 2.51103...",-34.162594
mp-1217529,-21.691357,"[[3.31349992 3.8320473 3.05919074] Tb, [2.751...","[[4.12533274 4.12533274 0.11396965] Tb, [4.128...",NaN
mp-1224010,-20.936572,"[[4.48186867 5.18938346 1.67596908] Hg, [2.492...","[[4.4927283 5.189859 1.67761523] Hg, [2.500...",-26.063581
mp-1660,-6.323225,"[[0. 0. 0.] Er, [1.7707945 1.7707945 1.7707945...","[[0. 0. 0.] Er, [1.75394916 1.75394916 1.75394...",-6.535980
mp-567981,-26.951303,"[[0. 0. 0.] Cr, [2.33653895 1.65218253 4.04700...","[[0. 0. 0.] Cr, [2.862799 2.862799 2.862799] G...",-27.375478
mp-755570,-24.104977,"[[0. 0. 0.] Li, [6.61492185e-07 2.13709915e+00...","[[0. 0. 0.] Li, [1.842065 1.06351846 2.16868...",-25.694250
mp-778768,-77.148564,"[[3.30972758 2.35004849 5.6094537 ] Li, [-2.31...","[[0. 6.4783065 2.419818 ] Li, [0. ...",NaN


In [45]:
reference_record[1]

mp-12052-r2SCAN ComputedStructureEntry - Er8 Sb6      (Er4Sb3)
Energy (Uncorrected)     = -440.8962 eV (-31.4926 eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -440.8962 eV (-31.4926 eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Er_3 06Sep2000', 'hash': 'b0fa0b2201af10ae95cd7fa622bf80ac', 'summary_stats': None}, {'titel': 'PAW_PBE Sb 06Sep2000', 'hash': '6e6f1eb0c5787752a1160664c2fa8425', 'summary_stats': None}]
  run_type               = r2SCAN
  is_hubbard             = False
  hubbards               = None
Data:
  oxide_type             = None
  aspherical             = True
  last_updated           = 2024-11-21 20:38:58.419384+00:00
  task_id                = mp-1975382
  material_id            = mp-12052
  oxidation_states       = {}
  license                = BY-C
  run_type               = R2SCAN

In [35]:
mp_id

'mp-778768'

In [36]:
reference_record[0] == reference_record[1]

True

In [33]:
reference_record[1]

mp-778768-GGA+U ComputedStructureEntry - Li2 Co3 Ni1 O8 (Li2Co3NiO8)
Energy (Uncorrected)     = -76.9811  eV (-5.4986  eV/atom)
Correction               = -12.9510  eV (-0.9251  eV/atom)
Energy (Final)           = -89.9321  eV (-6.4237  eV/atom)
Energy Adjustments:
  MP2020 anion correction (oxide): -5.4960   eV (-0.3926  eV/atom)
  MP2020 GGA/GGA+U mixing correction (Co): -4.9140   eV (-0.3510  eV/atom)
  MP2020 GGA/GGA+U mixing correction (Ni): -2.5410   eV (-0.1815  eV/atom)
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Li_sv 23Jan2001', 'hash': '4799bab014a83a07c654d7196c8ecfa9', 'summary_stats': None}, {'titel': 'PAW_PBE Co 06Sep2000', 'hash': 'b169bca4e137294d2ab3df8cbdd09083', 'summary_stats': None}, {'titel': 'PAW_PBE Ni_pv 06Sep2000', 'hash': '252a974c7f603c1f24e30fd36a0dba30', 'summary_stats': None}, {'titel': 'PAW_PBE O 08Apr2002', 'hash': '7a25bc5b9a5393f46600a4939d357982', 'summary_stats': None}]
  run_type               = GGA+U
  is_hubbard             = True

In [27]:
reference_energy[0].energy

-89.93209513